In [4]:
import sys
import os
import tqdm
import gc
import torch
import numpy as np
import pickle as pkl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm

module_path = os.path.abspath('..')
if module_path not in sys.path:
    sys.path.append(module_path)
    
from utils import ini_argparse, split_dataset
from dataset import *
from model import MinkUNetConvNeXtV2

import matplotlib
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import font_manager
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt

# reset the plot configurations to default
plt.rcdefaults()

from pathlib import Path
font_path = str(Path(matplotlib.get_data_path(), "fonts/ttf/cmr10.ttf"))
font_manager.fontManager.addfont(font_path)
prop = font_manager.FontProperties(fname=font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = prop.get_name()
plt.rcParams["axes.formatter.use_mathtext"] = True
params = {'mathtext.default': 'regular' }          
plt.rcParams.update(params)

In [5]:
# manually specify the GPUs to use
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#device = torch.device('cpu')

parser = ini_argparse()
args = parser.parse_args([])
#args.dataset_path = "/scratch/salonso/sparse-nns/faser/events_v3.5"
args.dataset_path = "/scratch/salonso/sparse-nns/faser/events_v3.5"
args.sets_path = "/scratch/salonso/sparse-nns/faser/events_v3.5/sets.pkl"
args.batch_size = 16
args.num_workers = 16

print("\n- Arguments:")
for arg, value in vars(args).items():
    print(f"  {arg}: {value}")
nb_gpus = len(args.gpus)
gpus = [int(gpu) for gpu in args.gpus]


- Arguments:
  train: True
  stage1: True
  dataset_path: /scratch/salonso/sparse-nns/faser/events_v3.5
  sets_path: /scratch/salonso/sparse-nns/faser/events_v3.5/sets.pkl
  load_seg: False
  eps: 1e-12
  chunk_size: 512
  batch_size: 16
  epochs: 50
  num_workers: 16
  lr: 0.0001
  accum_grad_batches: 1
  warmup_steps: 0
  cosine_annealing_steps: 0
  weight_decay: 0.05
  beta1: 0.9
  beta2: 0.999
  losses: ['focal', 'dice']
  save_dir: /scratch/salonso/sparse-nns/faser/deep_learning/faserDL
  name: v1
  log_every_n_steps: 50
  save_top_k: 1
  checkpoint_path: /scratch/salonso/sparse-nns/faser/deep_learning/faserDL/checkpoints
  checkpoint_name: v1
  load_checkpoint: None
  gpus: [0]


In [6]:
dataset = SparseFASERCALDataset(args)
print("- Dataset size: {} events".format(len(dataset)))
train_loader, valid_loader, test_loader = split_dataset(dataset, args, splits=[0.6, 0.1, 0.3], test=True)

- Dataset size: 144939 events
Loaded saved splits!


In [7]:
from torch.utils.data import DataLoader
from utils import collate_test

full_loader = DataLoader(
        dataset, batch_size=args.batch_size, num_workers=args.num_workers,
        shuffle=False, pin_memory=True, persistent_workers=True if args.num_workers > 0 else False,
        collate_fn=collate_test
    )

In [8]:
model = MinkUNetConvNeXtV2(in_channels=1, out_channels=4, D=3, args=args)
checkpoint = torch.load("/raid/monsals/faser/checkpoints_seg/seg_v6/last.ckpt", map_location='cpu')

# Remove the "model." prefix from the keys in the state_dict
state_dict = {key.replace("model.", ""): value for key, value in checkpoint['state_dict'].items()}
model.load_state_dict(state_dict, strict=True)

if device.type == 'cpu':
    model.replace_depthwise_with_channelwise()
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total trainable params model (total): {}".format(total_params))


FileNotFoundError: [Errno 2] No such file or directory: '/raid/monsals/faser/checkpoints_seg/seg_v6/last.ckpt'

In [ ]:
from utils import arrange_sparse_minkowski, arrange_truth
from sklearn.metrics import confusion_matrix as sklearn_confusion_matrix

def _arrange_batch(batch, device):
    batch_input, batch_input_global = arrange_sparse_minkowski(batch, device)
    batch_input_global = batch_input_global.to(device)
    target = arrange_truth(batch)
    return batch_input, batch_input_global, target

In [ ]:
model.eval()

conf_primlepton = {}
conf_seg = {}

count = 0
t = tqdm.tqdm(enumerate(full_loader), total=len(full_loader), disable=False)
for i, batch in t:
    torch.cuda.empty_cache()
    gc.collect()
        
    # Prepare input and target tensors
    batch_input, batch_input_global, target = _arrange_batch(batch, device)
    
    with torch.no_grad():
        batch_output = model(batch_input, batch_input_global)
    
    # pred
    out_primlepton = [torch.sigmoid(x).detach().cpu().numpy() for x in batch_output['out_primlepton'].decomposed_features]
    out_seg = [torch.softmax(x, dim=1).detach().cpu().numpy() for x in batch_output['out_seg'].decomposed_features]

    batch_size = len(out_primlepton)

    for batch_idx in range(batch_size):
        file_name = full_loader.dataset.data_files[count]
        data = np.load(file_name, allow_pickle=True)
        
        file_name = file_name.replace("events_v3.5", "events_v3.5_seg_results")
        
        out_primlepton_save = out_primlepton[batch_idx]
        out_seg_save = out_seg[batch_idx]
        
        np.savez_compressed(file_name,\
                            out_primlepton = out_primlepton_save,\
                            out_seg = out_seg_save
                           )
        count += 1
        
    del batch_input
    